In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import deadwood
import sklearn as sk
import scipy
import seaborn as sns

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Loader

In [20]:
import os
def load_all_datasets(directory):
    all_datasets = {}
    for root, dirs, files in os.walk(directory):
        for file in files:
            mat_data=scipy.io.loadmat(os.path.join(root,file))
            file_key = os.path.splitext(file)[0]
            df_x=pd.DataFrame(mat_data["X"])
            df_y=pd.DataFrame(mat_data["y"])

            duplicate_count = df_x.duplicated().sum()
            if duplicate_count > 0:
                print(f"Dataset '{file_key}' has {duplicate_count} duplicate rows (out of {df_x.shape[0]}. Removing them...")
                df_x_clean = df_x.drop_duplicates()
                df_y_clean = df_y.loc[df_x_clean.index]
                all_datasets[file_key]={
                    "X":df_x_clean.reset_index(drop=True),
                    "y":df_y_clean.reset_index(drop=True)
                }
            else:
                all_datasets[file_key]={
                    "X":pd.DataFrame(mat_data["X"]),
                    "y":pd.DataFrame(mat_data["y"])
                }
    return all_datasets


In [22]:
all_datasets = load_all_datasets("C:\\Users\\andrz\\Downloads\\dev_proj2_data")


Dataset 'annthyroid' has 138 duplicate rows (out of 7200. Removing them...
Dataset 'breastw' has 234 duplicate rows (out of 683. Removing them...
Dataset 'cardio' has 9 duplicate rows (out of 1831. Removing them...
Dataset 'glass' has 1 duplicate rows (out of 214. Removing them...
Dataset 'letter' has 2 duplicate rows (out of 1600. Removing them...
Dataset 'mammography' has 3335 duplicate rows (out of 11183. Removing them...
Dataset 'satimage-2' has 2 duplicate rows (out of 5803. Removing them...
Dataset 'thyroid' has 116 duplicate rows (out of 3772. Removing them...
Dataset 'vowels' has 4 duplicate rows (out of 1456. Removing them...
dict_keys(['annthyroid', 'arrhythmia', 'breastw', 'cardio', 'glass', 'letter', 'lympho', 'mammography', 'musk', 'pendigits', 'satellite', 'satimage-2', 'shuttle', 'speech', 'thyroid', 'vertebral', 'vowels', 'wine'])


# Scoring functions

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score


def get_unified_outlier_scores(model, X, method_name):
    """
    Standardizes the raw outputs of different sklearn algorithms so that
    higher scores always point towards the point being more anomalous.
    """
    if method_name == "Isolation Forest":
        return -model.score_samples(X)

    elif method_name == "One-Class SVM":
        return -model.score_samples(X)

    elif method_name == "Local Outlier Factor":
        return -model.negative_outlier_factor_

    elif method_name == "DBSCAN":
        labels = model.labels_
        return np.where(labels == -1, 1.0, 0.0)
    else:
        raise ValueError(f"Unknown method name: {method_name}")

def threshold_by_top_k(scores, contamination_rate):
    """
    Flags the top-k% highest scores as outliers (1), and the rest as inliers (0).
    Useful for datasets where ground-truth contamination is known or estimated.
    """
    if contamination_rate <= 0 or contamination_rate >= 1:
        raise ValueError("Contamination rate must be between 0 and 1 exclusively.")

    n_samples = len(scores)
    k = int(np.ceil(contamination_rate * n_samples))
    threshold = np.partition(scores, -k)[-k]
    binary_preds = np.where(scores >= threshold, 1, 0)
    return binary_preds


def compute_evaluation_metrics(y_true, y_scores, y_pred=None, contamination=None):
    """
    Computes standard evaluation metrics based on ground truth and model outputs.
    If continuous scores are passed without binary predictions, it auto-thresholds
    using the provided contamination rate.
    """
    if y_pred is None:
        if contamination is None:
            raise ValueError("Must provide either 'y_pred' or a 'contamination' rate to calculate discrete metrics.")
        y_pred = threshold_by_top_k(y_scores, contamination)
    metrics = {
        "AUC-ROC": roc_auc_score(y_true, y_scores) if len(np.unique(y_true)) > 1 else np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

    return metrics, y_pred